In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
class Generator(nn.Module):
    def __init__(self, input_dim, label_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim + label_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, z, labels):
        z = torch.cat((z, labels), dim=1)
        return self.model(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim, label_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim + label_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x, labels):
        x = torch.cat((x, labels), dim=1)
        return self.model(x)


In [5]:
def initialize_models(input_dim, label_dim, output_dim):
    generator = Generator(input_dim, label_dim, output_dim)
    discriminator = Discriminator(output_dim, label_dim)
    return generator, discriminator

def initialize_optimizers(generator, discriminator, lr=0.0001):
    optimizer_g = optim.RMSprop(generator.parameters(), lr=lr)
    optimizer_d = optim.RMSprop(discriminator.parameters(), lr=lr)
    return optimizer_g, optimizer_d


In [6]:
def train_wgan(generator, discriminator, optimizer_g, optimizer_d, data, labels, num_epochs=1000):
    for epoch in range(num_epochs):
        for i in range(len(data)):
            # Train Discriminator
            optimizer_d.zero_grad()
            real_data = torch.tensor(data[i], dtype=torch.float32).unsqueeze(0)
            real_labels = torch.tensor(labels[i], dtype=torch.float32).unsqueeze(0)
            
            z = torch.randn(real_data.size(0), 100)  # Random noise
            generated_data = generator(z, real_labels)

            d_loss = -torch.mean(discriminator(real_data, real_labels)) + torch.mean(discriminator(generated_data.detach(), real_labels))
            d_loss.backward()
            optimizer_d.step()

            # Train Generator
            optimizer_g.zero_grad()
            g_loss = -torch.mean(discriminator(generated_data, real_labels))
            g_loss.backward()
            optimizer_g.step()
            print(f'Epoch [{epoch}/{num_epochs}], D Loss: {d_loss.item()}, G Loss: {g_loss.item()}')

In [7]:
def generate_samples(generator, num_samples, label):
    z = torch.randn(num_samples, 100)  # Random noise
    label_tensor = torch.tensor(np.array([[label]] * num_samples), dtype=torch.float32)
    return generator(z, label_tensor).detach().numpy()

In [1]:
names = np.array(
    [
        "Length",
        "Width",
        "Size",
        "Conc",
        "Conc1",
        "Asym",
        "M3Long",
        "M3Trans",
        "Alpha",
        "Dist",
        "class",
    ]
)
data = pd.read_csv("data/magic04.data", header=None)
data.columns = names
data.head()

NameError: name 'np' is not defined

In [12]:
from sklearn.preprocessing import QuantileTransformer
X = data[names[:-1]].values
X = np.abs(X)
# метки классов
labels = data[names[-1]].values
y = np.ones((len(labels), 1))
y[labels == "h"] = 0

transformer = QuantileTransformer(output_distribution='normal')

X_qt = transformer.fit_transform(X)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_qt, y, stratify=y, test_size=0.5, shuffle=True, random_state=11
)
print("X_train:",X_train)
print("X_test:",X_test)

X_train: [[ 0.20939934  0.26039373  0.7349225  ... -1.35291912  0.57756652
  -1.60782057]
 [ 1.60835656  2.14210624  1.98481412 ...  1.16448676  1.54056901
   0.07784221]
 [ 0.19846121  0.3488263  -0.05481026 ... -0.50572951 -0.9211557
   0.690594  ]
 ...
 [-0.1789733   0.35862748  0.42202559 ... -0.18494837  0.34976835
  -1.49468758]
 [ 0.83226266  0.63746528  1.0545915  ...  0.39800762 -0.53482694
   1.18331607]
 [-0.46954249 -0.38068457 -0.41426686 ... -0.10077911 -0.65627682
  -0.45367005]]
X_test: [[-1.06732007e-01 -1.96409758e+00 -3.80938743e-01 ... -1.16083116e+00
  -5.26246880e-01  8.17668973e-01]
 [ 2.22603060e+00  1.91104408e+00  2.60095793e+00 ...  1.04477664e+00
   1.45234932e+00  1.89565958e+00]
 [-1.84234362e+00 -9.60655618e-01 -1.22509688e+00 ...  1.18439284e-01
   1.32663937e-01  2.78490839e-02]
 ...
 [-1.33637815e+00 -1.60117882e+00 -1.43197089e+00 ... -1.87266737e+00
   1.48062894e+00 -1.42165853e+00]
 [ 8.15076279e-01  8.76691988e-01  1.46046314e+00 ...  3.13440598e-

In [18]:

# Initialize models, optimizers
generator, discriminator = initialize_models(input_dim=100, label_dim=1, output_dim=X_train.shape[1])
optimizer_g, optimizer_d = initialize_optimizers(generator, discriminator)

# Train WGAN
train_wgan(generator, discriminator, optimizer_g, optimizer_d, X_train, y_train,num_epochs=5)

# Generate samples
samples = generate_samples(generator, num_samples=5, label=0)
print("Generated Samples:\n", samples)

Epoch [0/5], D Loss: 1.104060411453247, G Loss: 2.474985122680664
Generated Samples:
 [[ 1.0290722  -0.03477856  0.5900761  -0.5065583  -0.83922595  1.3026719
   0.53227675  0.47000983 -2.612598    1.5416296 ]
 [ 0.9928656   0.38948944  0.8149481  -0.813035   -0.90747195  0.9161349
   0.45694676  0.68051827 -2.2642908   1.4266976 ]
 [ 0.43216902  0.30208912  0.37227675 -0.5261276  -0.7850638   0.45093906
   0.13685118  0.60841787 -1.7953503   0.6996183 ]
 [ 0.12933265 -0.45885614  0.18410647 -0.06507155 -0.10114029  0.15235877
  -0.19481401 -0.20607434 -1.3124732   0.31388414]
 [ 0.73866004  0.19250098  0.630611   -0.56737673 -0.88739216  0.41061264
   0.49934587  0.8833941  -1.9907987   1.1482372 ]]
